## seed_dim_geo
Seeds `silver.dim_geo` (CBSA grain) from the committed crosswalk reference data on the
`reference` Volume (`CROSSWALKS`): `cbsa_master.csv` (OMB 2023 universe) is the row source;
`zillow_region_to_cbsa.csv` supplies `zillow_region_id`; `household_rank` is enriched from
`bronze.realtor_metro_monthly` (latest month per CBSA); `census_region` is derived from
`primary_state` (US Census 4-region map; territories → null); `cbsa_population` is summed from
`bronze.fema_nri_counties` over the `county_to_cbsa.csv` bridge (the shared weight for the
weather Silver/Gold intensive rollups).

Idempotent **MERGE on `cbsa_code`** — `geo_key` is GENERATED ALWAYS AS IDENTITY and is
omitted from INSERT (Delta assigns it), and MERGE keeps it **stable** across re-seeds so
fact foreign keys never break. Upload the CSVs to the Volume first (rare reference refresh):
`cbsa_master.csv`, `zillow_region_to_cbsa.csv`, **`county_to_cbsa.csv`**.

In [ ]:
%run "../libs/notebook_init"

In [ ]:
# notebook_init injects SILVER, BRONZE, CROSSWALKS, F, spark. Window is not injected.
from pyspark.sql import Window

# US Census region of each state (4 regions over the 50 states + DC). Territories (e.g. PR)
# are deliberately absent -> their CBSAs get census_region = null (Census places them outside
# the four regions). Inline constant, mirroring STATE_NAME_TO_POSTAL in build_crosswalk.py.
STATE_TO_CENSUS_REGION = {
    # Northeast
    "CT": "Northeast", "ME": "Northeast", "MA": "Northeast", "NH": "Northeast",
    "RI": "Northeast", "VT": "Northeast", "NJ": "Northeast", "NY": "Northeast", "PA": "Northeast",
    # Midwest
    "IL": "Midwest", "IN": "Midwest", "MI": "Midwest", "OH": "Midwest", "WI": "Midwest",
    "IA": "Midwest", "KS": "Midwest", "MN": "Midwest", "MO": "Midwest", "NE": "Midwest",
    "ND": "Midwest", "SD": "Midwest",
    # South
    "DE": "South", "FL": "South", "GA": "South", "MD": "South", "NC": "South", "SC": "South",
    "VA": "South", "DC": "South", "WV": "South", "AL": "South", "KY": "South", "MS": "South",
    "TN": "South", "AR": "South", "LA": "South", "OK": "South", "TX": "South",
    # West
    "AZ": "West", "CO": "West", "ID": "West", "MT": "West", "NV": "West", "NM": "West",
    "UT": "West", "WY": "West", "AK": "West", "CA": "West", "HI": "West", "OR": "West",
    "WA": "West",
}

# Reference CSVs from the reference Volume (all STRING; cast household_rank to INT).
cbsa = spark.read.option("header", True).csv(f"{CROSSWALKS}cbsa_master.csv")
xwalk = (
    spark.read.option("header", True).csv(f"{CROSSWALKS}zillow_region_to_cbsa.csv")
    .where("cbsa_code IS NOT NULL AND cbsa_code <> ''")
)

# One zillow_region_id per CBSA (collisions are rare; take the min deterministically).
region_by_cbsa = xwalk.groupBy("cbsa_code").agg(F.min("region_id").alias("zillow_region_id"))

# household_rank = the most recent month's value per CBSA from Bronze Realtor.
realtor = spark.table(f"{BRONZE}.realtor_metro_monthly").where(
    "household_rank IS NOT NULL AND household_rank <> ''"
)
w = Window.partitionBy("cbsa_code").orderBy(F.col("month_date_yyyymm").desc())
hh = (
    realtor.withColumn("rn", F.row_number().over(w)).where("rn = 1")
    .select("cbsa_code", F.col("household_rank").cast("double").cast("int").alias("household_rank"))
)

# cbsa_population = sum of FEMA NRI county population over the CBSA's counties. NRI is county
# grain (stcofips); the county->CBSA bridge maps each county to its CBSA. NRI counties with no
# CBSA (rural) drop on the inner join (CBSA-footprint coverage). Trim before cast — Bronze
# keeps NCEI/NRI source padding.
county_bridge = spark.read.option("header", True).csv(f"{CROSSWALKS}county_to_cbsa.csv")
nri = spark.table(f"{BRONZE}.fema_nri_counties").where("population IS NOT NULL AND population <> ''")
cbsa_pop = (
    nri.join(county_bridge, "stcofips", "inner")
       .groupBy("cbsa_code")
       .agg(F.sum(F.trim(F.col("population")).cast("long")).alias("cbsa_population"))
)

# census_region lookup as a small DataFrame so it joins on primary_state (left join -> null
# for territories absent from the dict).
region_lookup = spark.createDataFrame(
    list(STATE_TO_CENSUS_REGION.items()), ["primary_state", "census_region"]
)

staging = (
    cbsa.join(region_by_cbsa, "cbsa_code", "left")
        .join(hh, "cbsa_code", "left")
        .join(cbsa_pop, "cbsa_code", "left")
        .join(region_lookup, "primary_state", "left")
        .select("cbsa_code", "cbsa_title", "cbsa_type", "zillow_region_id",
                "primary_state", "state_list", "household_rank",
                "census_region", "cbsa_population")
)
staging.createOrReplaceTempView("dim_geo_staging")

# geo_key (IDENTITY) is omitted from INSERT so Delta assigns it; MERGE keeps it stable.
spark.sql(f"""
    MERGE INTO {SILVER}.dim_geo t USING dim_geo_staging s ON t.cbsa_code = s.cbsa_code
    WHEN MATCHED THEN UPDATE SET
        t.cbsa_title=s.cbsa_title, t.cbsa_type=s.cbsa_type,
        t.zillow_region_id=s.zillow_region_id, t.primary_state=s.primary_state,
        t.state_list=s.state_list, t.household_rank=s.household_rank,
        t.census_region=s.census_region, t.cbsa_population=s.cbsa_population,
        t.updated_ts=current_timestamp()
    WHEN NOT MATCHED THEN INSERT
        (cbsa_code, cbsa_title, cbsa_type, zillow_region_id, primary_state, state_list,
         household_rank, census_region, cbsa_population, inserted_ts, updated_ts)
        VALUES (s.cbsa_code, s.cbsa_title, s.cbsa_type, s.zillow_region_id, s.primary_state,
                s.state_list, s.household_rank, s.census_region, s.cbsa_population,
                current_timestamp(), current_timestamp())
""")

dim_geo = spark.table(f"{SILVER}.dim_geo")
total = dim_geo.count()
with_zillow = dim_geo.where("zillow_region_id IS NOT NULL").count()
with_rank = dim_geo.where("household_rank IS NOT NULL").count()
with_region = dim_geo.where("census_region IS NOT NULL").count()
with_pop = dim_geo.where("cbsa_population IS NOT NULL").count()
print(f"seed_dim_geo: dim_geo rows = {total:,} (zillow_region_id: {with_zillow:,}, "
      f"household_rank: {with_rank:,}, census_region: {with_region:,}, "
      f"cbsa_population: {with_pop:,})")